<a href="https://colab.research.google.com/github/karsarobert/LLM2026/blob/main/ChatGpt2025_07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Chat GPT és más nagy nyelvi modellek alkalmazása
##PTE Gépi tanulás
###7. gyakorlat: OpenAI-kompatibilis API és eszközhívások (Tool Calling)
2026. március 16.

Ez a notebook az **OpenAI-kompatibilis API** használatát mutatja be, majd bevezeti az **eszközhívások (tool calling)** fogalmát.

Eddig a Google Gemini natív API-ját használtuk. Most egy **iparági szabványt** ismerünk meg:
az OpenAI chat completions formátumot, amelyet ma már szinte minden LLM szolgáltató támogat.

**API szolgáltató:** [OpenRouter](https://openrouter.ai) – egyetlen API kulccsal sok modellt érünk el, köztük ingyeneseket is.

**Modell:** `openai/gpt-oss-120b:free`

**A gyakorlat témái:**
1. Mi az OpenAI-kompatibilis API?
2. OpenRouter bemutatása és beállítása
3. Első API hívás – chat completion
4. Szerepek: system, user, assistant
5. JSON mód
6. Eszközhívás (Tool Calling) – elmélet
7. Első eszköz: időjárás lekérdezés
8. Több eszköz együtt
9. Feladatok

---
## 1. Mi az OpenAI-kompatibilis API?

Az OpenAI bevezette a **chat completions** API formátumot, amely mára **iparági szabvánnyá** vált.

| Tulajdonság | Gemini natív API | OpenAI-kompatibilis API |
|---|---|---|
| Python csomag | `google-generativeai` | `openai` |
| Fő metódus | `model.generate_content()` | `client.chat.completions.create()` |
| Üzenet formátum | `contents=[...]` | `messages=[{"role": ..., "content": ...}]` |
| Támogató szolgáltatók | csak Google | OpenAI, OpenRouter, Groq, Together, Ollama, stb. |

**Előnye:** egyszer megtanulod → bármely szolgáltatóval működik. Csak a `base_url` és `api_key` változik!

---
## 2. OpenRouter bemutatása

Az [OpenRouter](https://openrouter.ai) egy **API gateway**: egyetlen kulccsal sok különböző modellt érhetsz el.

**Miért jó?**
- Ingyenes modellek is elérhetőek (`:free` végződés)
- OpenAI-kompatibilis végpont → ugyanaz a kód működik, mint az OpenAI-nál
- Nem kell külön regisztrálni minden szolgáltatóhoz

**API kulcs beszerzése:**
1. Regisztráció: [openrouter.ai](https://openrouter.ai)
2. Keys menü → Create Key
3. A kulcsot másold be a Colab titkos tárolójába `OPENROUTER_API_KEY` néven

**A mai modellünk:** `openai/gpt-oss-120b:free` – az OpenAI nyílt súlyú modellje, 120 milliárd paraméterrel.

### 1. lépés – Az `openai` csomag telepítése

Az `openai` Python csomag nem csak az OpenAI saját szolgáltatásához használható –
a `base_url` paraméterrel **bármely kompatibilis szolgáltatóra** irányítható.

In [ ]:
!pip install openai --quiet

print('openai csomag telepítve')

### 2. lépés – API kulcs betöltése

A Colab `userdata` tárolójából olvassuk a kulcsot – ugyanúgy, mint a Gemini kulcsnál.
Soha ne írjuk a kulcsot közvetlenül a kódba!

In [ ]:
try:
    from google.colab import userdata
    api_key = userdata.get('OPENROUTER_API_KEY')   # Colab titkos tárolóból
except Exception:
    import getpass
    api_key = getpass.getpass('OpenRouter API kulcs: ')   # manuális megadás

print('API kulcs betöltve')

API kulcs betöltve


In [ ]:
try:
    from google.colab import userdata
    api_key = userdata.get('GROQ_API_KEY')   # Colab titkos tárolóból
except Exception:
    import getpass
    api_key = getpass.getpass('Groq API kulcs: ')   # manuális megadás

print('API kulcs betöltve')

API kulcs betöltve


### 3. lépés – Az OpenAI kliens inicializálása OpenRouter-re irányítva

A kulcs: a `base_url` paraméter. Ez mondja meg a kliensnek, hogy ne az OpenAI szerverére,
hanem az OpenRouter szerverére küldje a kéréseket.

```
OpenAI saját:   https://api.openai.com/v1
OpenRouter:     https://openrouter.ai/api/v1
Groq:           https://api.groq.com/openai/v1
Ollama (helyi): http://localhost:11434/v1
```

Ugyanaz a kód – csak a cím változik!

In [ ]:
from openai import OpenAI   # az openai csomag fő osztálya

client = OpenAI(
    base_url='https://openrouter.ai/api/v1',   # OpenRouter végpont
    api_key=api_key                              # a mi kulcsunk
)

MODELL = 'openai/gpt-oss-120b:free'   # ingyenes modell

print(f'Kliens kész, modell: {MODELL}')

Kliens kész, modell: openai/gpt-oss-120b:free


In [ ]:
from openai import OpenAI   # az openai csomag fő osztálya

client = OpenAI(
    base_url='https://api.groq.com/openai/v1/',   # OpenRouter végpont
    api_key=api_key                              # a mi kulcsunk
)

MODELL = 'openai/gpt-oss-120b'   # ingyenes modell

print(f'Kliens kész, modell: {MODELL}')

Kliens kész, modell: openai/gpt-oss-120b


In [ ]:
#

---
## 3. Első API hívás – Chat Completion

Az OpenAI API **üzenet-alapú**: egy `messages` listát küldünk, ahol minden üzenet
egy szótár `role` és `content` kulcsokkal.

```python
messages = [
    {"role": "user", "content": "Szia!"}
]
```

A válasz objektum struktúrája:
```
response.choices[0].message.content   →  a szöveges válasz
response.choices[0].message.role      →  "assistant"
```

### 1. lépés – Egyszerű kérdés küldése

A `client.chat.completions.create()` metódust használjuk.
Két kötelező paraméter: `model` és `messages`.

In [ ]:
valasz = client.chat.completions.create(
    model=MODELL,
    messages=[
        {'role': 'user', 'content': 'Mi Magyarország fővárosa? Válaszolj egy mondatban.'}
    ]
)

print(valasz.choices[0].message.content)

Budapest Magyarország fővárosa.


### 2. lépés – A válasz objektum felépítése

Nézzük meg a teljes válasz struktúráját. Ez segít megérteni, milyen adatokat kapunk vissza.

In [ ]:
print('Típus:', type(valasz))
print('Modell:', valasz.model)
print('Szerepkör:', valasz.choices[0].message.role)
print('Tartalom:', valasz.choices[0].message.content)
print('Token használat:', valasz.usage)

Típus: <class 'openai.types.chat.chat_completion.ChatCompletion'>
Modell: openai/gpt-oss-120b
Szerepkör: assistant
Tartalom: Budapest Magyarország fővárosa.
Token használat: CompletionUsage(completion_tokens=75, prompt_tokens=87, total_tokens=162, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=58, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=2.9359261180000003, prompt_time=0.004773196, completion_time=0.180287809, total_time=0.185061005)


In [69]:
valasz

ChatCompletion(id='chatcmpl-b44614df-e2fb-461e-8445-a1649c4ce160', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_22b0f045-1817-4f01-805a-dae6034903e7', function=Function(arguments='{"varos":"Pécs"}', name='idojaras_lekerdez'), type='function')], reasoning='User asks current weather in Pécs. Use function.'))], created=1773669177, model='openai/gpt-oss-120b', object='chat.completion', service_tier='on_demand', system_fingerprint='fp_6b677c2caf', usage=CompletionUsage(completion_tokens=48, prompt_tokens=154, total_tokens=202, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=13, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=1.09544955, prompt_time=0.00810648, completion_time=0.12169025, total_

### 3. lépés – Segédfüggvény az egyszerűbb hívásokhoz

Mivel a `client.chat.completions.create(...)` és a `valasz.choices[0].message.content`
elég hosszú, készítünk egy rövid segédfüggvényt, amit a notebook hátralevő részében használunk.

In [ ]:
def kerdez(uzenet, system_prompt=None):
    """Egyszerű chat completion hívás – visszaadja a szöveges választ."""
    messages = []
    if system_prompt:
        messages.append({'role': 'system', 'content': system_prompt})
    messages.append({'role': 'user', 'content': uzenet})

    valasz = client.chat.completions.create(
        model=MODELL,
        messages=messages
    )
    return valasz.choices[0].message.content

# Teszt
print(kerdez('Melyik a legnagyobb magyar tó?'))

A legnagyobb magyar tó a **Balaton**.  

- **Terület**: körülbelül 592 km² (a felszíne a legnagyobb a magyar tavak között).  
- **Hossza**: kb. 77 km, szélessége 10–14 km.  
- **Mélysége**: átlagosan 3,2 m, legmélyebb pontja 12,2 m.

A Balaton “magyar tengerként” is ismert, népszerű üdülőhely, vízi sportok és turizmus központja. (Bár a mesterséges Tisza‑tó (Tisza-tó) is nagy, a Balaton a természetes, és összterületében is nagyobb.)


---
## 4. Szerepek: system, user, assistant

Az OpenAI formátumban **három szerepkör** létezik:

| Szerepkör | Ki? | Mire jó? |
|---|---|---|
| `system` | A fejlesztő | Viselkedés, személyiség, szabályok meghatározása |
| `user` | A felhasználó | Kérdések, utasítások |
| `assistant` | Az LLM | Korábbi válaszok (többkörös beszélgetéshez) |

A `system` prompt egy **láthatatlan utasítás** – a felhasználó nem látja, de az LLM követi.

### 1. lépés – System prompt nélkül

Először system prompt nélkül kérdezünk – az LLM az alapértelmezett viselkedését mutatja.

In [ ]:
print(kerdez('Adj 3 tippet a hatékony tanuláshoz.'))

**3 tipp a hatékony tanuláshoz**

1. **Aktív ismétlés (spaced repetition)**
   - A tanult anyagot ne csak egyszer olvasd át, hanem ismételd meg időközönként növekvő időintervallumokban (pl. 1 nap, 3 nap, 1 hét).  
   - Használhatsz kártyákat (Anki, Quizlet) vagy egyszerű jegyzeteket, amelyek segítenek a memória megerősítésében.  
   - Ez a módszer a hosszú távú memóriába való beágyazást segíti, és csökkenti az „elfelejtő‑görbe” hatását.

2. **Tanulj különböző módokon (multimodális tanulás)**
   - Kombináld az írott anyagokat (könyvek, jegyzetek) vizuális elemekkel (diagramok, infografikák), hallható anyagokkal (podcastok, magyarázó videók) és aktív gyakorlással (feladatok, szimulációk).  
   - A különböző érzékszervi csatornák egyidejű aktiválása mélyebb megértést és jobb rögzülést eredményez.  
   - Példa: egy elméleti fejezet után készíts egy rövid vázlatot, rajzolj hozzá egy folyamatábrát, majd magyarázd el hangosan egy barátodnak vagy felvételként rögzítsd.

3. **Állíts fel konkrét

### 2. lépés – System prompt: motivációs edző

Most a system promptban megmondjuk: **motivációs edzőként** válaszoljon, röviden és lelkesen.

In [ ]:
print(kerdez(
    'Adj 3 tippet a hatékony tanuláshoz.',
    system_prompt='Motivációs edző vagy. Röviden, lelkesen válaszolj, magyarul. Maximum 3 mondat tippenként.'
))

1. **Tervezd meg a tanulási blokkokat!** Állíts fel konkrét célokat óránként, majd tarts szünetet 5‑10 percig, hogy az agyad felfrissüljön. A rendszeres időbeosztás segít a fókusz megtartásában és a tudás mélyebb rögzülésében.  

2. **Használj aktív tanulási módszereket!** Készíts saját jegyzeteket, magyarázd el hangosan a tananyagot, vagy tanítsd el egy barátodnak. Az aktív feldolgozás sokkal erősebben rögzíti az információt, mint a passzív olvasás.  

3. **Alakíts ki megfelelő környezetet!** Szabadíts fel egy csendes, rendezett helyet, kapcsold ki a zavaró értesítéseket, és tartsd kéznél a szükséges anyagokat. Egy koncentrációt elősegítő környezet segít, hogy a tanulás hatékony és élvezetes legyen.


### 3. lépés – System prompt: szakértő asszisztens

Ugyanaz a kérdés, de most **tudományos szakértő** stílusban kérjük a választ.

In [ ]:
print(kerdez(
    'Adj 3 tippet a hatékony tanuláshoz.',
    system_prompt='Kognitív pszichológus vagy. Hivatkozz kutatási eredményekre. Válaszolj magyarul.'
))

**3 tudományosan megalapozott tipp a hatékony tanuláshoz**

| # | Tipp | Miért működik? (kognitív‑pszichológiai magyarázat) | Kulcsfontosságú kutatási eredmények |
|---|------|-----------------------------------------------------|--------------------------------------|
| 1 | **Időközi ismétlés (spaced repetition)** – oszd szét a tanulási szakaszokat a napok vagy hetek folyamán. | Az információk hosszú távú tárolása nem a tömör „maraton” tanulásból, hanem a rendszeres, időben elosztott gyakorlásból ered. Az időközi ismétlés erősíti a kontextuális és epizodikus memóriatrace‑eket, miközben csökkenti a „közvetlen felejtés” görbéjét (Ebbinghaus, 1885). | Cepeda, Pashler, Vul, Wixted & Rohrer (2006) meta‑analízise 254 tanulmányra mutatta, hogy a “spacing effect” átlagosan **23 %‑kal** javítja a visszakeresési pontosságot a tömör gyakorlással szemben. |
| 2 | **Visszakeresés (retrieval practice)** – minden tanulási blokk végén próbáld visszahívni a tanult anyagot, ne csak újraolvasd. | A viss

---
## 5. JSON mód

Az OpenAI-kompatibilis API-ban is kérhetünk **strukturált JSON kimenetet** –
ez a korábbi gyakorlatokon tanult `response_mime_type` megfelelője.

Fontos különbségek a Gemini natív API-hoz képest:
- A `response_format` mellett a **promptban is kérni kell** a JSON kimenetet
- Az ingyenes/nyílt modellek nem mindig tartják be a JSON módot – néha Markdown kódblokkba csomagolják a választ (` ```json ... ``` `)
- Ezért szükségünk van egy **robusztus JSON kinyerő** függvényre (ismerjük a 6. gyakorlatból!)

### 1. lépés – Robusztus JSON kinyerő függvény

Ezt a függvényt a 6. gyakorlaton már megismertük. Három módszert próbál sorban:
1. Közvetlen `json.loads()` – ha tiszta JSON jött
2. Markdown kódblokk eltávolítása – ha ` ```json ... ``` ` jelölők vannak
3. Nyers `{` ... `}` keresés – ha extra szöveg van a JSON körül

In [ ]:
import json

def json_kinyerese(szoveg):
    """Robusztus JSON kinyerés – kezeli a Markdown kódblokkot és extra szöveget is."""
    szoveg = szoveg.strip()
    # 1. kísérlet: közvetlen parse
    try:
        return json.loads(szoveg)
    except json.JSONDecodeError:
        pass
    # 2. kísérlet: Markdown kódblokk (```json ... ```)
    if '```json' in szoveg:
        e = szoveg.find('```json') + 7
        v = szoveg.find('```', e)
        if v != -1:
            try:
                return json.loads(szoveg[e:v].strip())
            except json.JSONDecodeError:
                pass
    # 3. kísérlet: első { ... utolsó } közötti rész
    e = szoveg.find('{')
    v = szoveg.rfind('}') + 1
    if e != -1 and v > e:
        try:
            return json.loads(szoveg[e:v])
        except json.JSONDecodeError:
            pass
    # 3b. kísérlet: lista [ ... ]
    e = szoveg.find('[')
    v = szoveg.rfind(']') + 1
    if e != -1 and v > e:
        try:
            return json.loads(szoveg[e:v])
        except json.JSONDecodeError:
            pass
    return None

print('json_kinyerese() függvény kész')

json_kinyerese() függvény kész


### 2. lépés – JSON válasz kérése

A `response_format={"type": "json_object"}` paraméterrel kérjük a JSON kimenetet.
A nyers választ kiírjuk – láthatjuk, hogy a modell betartotta-e a formátumot.

In [ ]:
valasz = client.chat.completions.create(
    model=MODELL,
    response_format={'type': 'json_object'},   # JSON mód
    messages=[
        {'role': 'system', 'content': 'Mindig válaszolj kizárólag JSON formátumban, semmilyen egyéb szöveget ne adj hozzá.'},
        {'role': 'user', 'content': 'Add meg Pécs 3 nevezetességét egy JSON listában, minden elem: {"nev": "...", "leiras": "..."}.'}
    ]
)

nyers_valasz = valasz.choices[0].message.content
print('--- Nyers válasz (első 200 karakter) ---')
print(nyers_valasz[:200])
print()

# Robusztus kinyerés – akkor is működik, ha Markdown blokkba csomagolja
eredmeny = json_kinyerese(nyers_valasz)
if eredmeny:
    print('--- Kinyert JSON ---')
    print(json.dumps(eredmeny, indent=2, ensure_ascii=False))
else:
    print('HIBA: nem sikerült JSON-t kinyerni a válaszból')

--- Nyers válasz (első 200 karakter) ---
[
{"nev":"Pécsi Székesegyház","leiras":"A 11. századi román stílusú székesegyház, amely a város egyik legjelentősebb vallási épülete."},{"nev":"Zsolnay Porcelánmúzeum","leiras":"A híres Zsolnay család

--- Kinyert JSON ---
[
  {
    "nev": "Pécsi Székesegyház",
    "leiras": "A 11. századi román stílusú székesegyház, amely a város egyik legjelentősebb vallási épülete."
  },
  {
    "nev": "Zsolnay Porcelánmúzeum",
    "leiras": "A híres Zsolnay család kerámia- és porcelánművészetét bemutató múzeum, amely Pécs kulturális örökségének központja."
  },
  {
    "nev": "Korai keresztény temető (UNESCO)",
    "leiras": "A 4‑5. századi keresztény temető, amely a város egykori keresztény közösségét őrzi, és UNESCO Világörökség része."
  }
]


### 3. lépés – Az eredmény feldolgozása

A `json_kinyerese()` dict-et vagy listát ad vissza – ugyanúgy dolgozzuk fel, mint eddig.

In [ ]:
# Az eredmény struktúrától függ – az LLM jellemzően listát vagy dict-et ad
print('Típus:', type(eredmeny))

if isinstance(eredmeny, dict):
    print('Kulcsok:', list(eredmeny.keys()))
elif isinstance(eredmeny, list):
    print(f'{len(eredmeny)} elem a listában')
    for elem in eredmeny:
        print(f'  - {elem.get("nev", "?")}')


Típus: <class 'list'>
3 elem a listában
  - Pécsi Székesegyház
  - Zsolnay Porcelánmúzeum
  - Korai keresztény temető (UNESCO)


---
## 6. Eszközhívás (Tool Calling) – elmélet

### A probléma

Az LLM **nem tud valós időben adatot lekérni**: nem ismeri az aktuális időjárást,
nem tud adatbázist kezelni, nem tud e-mailt küldeni. A tudása a tanítási adatokra korlátozódik.

### A megoldás: eszközhívás (Tool Calling / Function Calling)

Az LLM-nek **eszközöket (tools)** definiálunk – ezek leírják, milyen függvények állnak rendelkezésre.
Az LLM a beszélgetés alapján **eldönti**, melyik eszközt kell meghívni és milyen paraméterekkel.

### A folyamat

```
1. Felhasználó: "Milyen idő van Pécsett?"
        ↓
2. LLM gondolkodik: "Ehhez az idojaras_lekerdez eszköz kell, varos=Pécs"
        ↓
3. LLM visszaad: tool_call (NEM szöveges válasz!)
        ↓
4. A MI KÓDUNK végrehajtja a függvényt: idojaras_lekerdez("Pécs") → {"hofok": 18, ...}
        ↓
5. Az eredményt visszaküldjük az LLM-nek
        ↓
6. LLM válaszol: "Pécsett jelenleg 18°C van, napos idő."
```

**Fontos:** az LLM **nem hajtja végre** a függvényt – csak megmondja, mit kellene meghívni!
A végrehajtás a mi kódunkban történik.

---
## 7. Első eszköz: időjárás lekérdezés

Készítünk egy **szimulált** időjárás lekérdező eszközt.
Szimulált, mert nem hívunk valós API-t – a lényeg a tool calling mechanizmus megértése.

### 1. lépés – A szimulált függvény megírása

Ez a függvény szimulálja az időjárás API-t: néhány város adatait egy szótárban tároljuk.
Valós alkalmazásban itt lenne a `requests.get()` hívás egy időjárás API felé.

In [ ]:
import json

def idojaras_lekerdez(varos):
    """Szimulált időjárás API – visszaadja a város időjárását."""
    adatok = {
        'Pécs':     {'hofok': 18, 'allapot': 'napos',   'paratartalom': 45},
        'Budapest': {'hofok': 15, 'allapot': 'felhős',  'paratartalom': 60},
        'Debrecen': {'hofok': 12, 'allapot': 'esős',    'paratartalom': 80},
        'Szeged':   {'hofok': 20, 'allapot': 'napos',   'paratartalom': 40},
    }
    if varos in adatok:
        return json.dumps(adatok[varos], ensure_ascii=False)
    else:
        return json.dumps({'hiba': f'{varos} nem található'}, ensure_ascii=False)

# Teszt
print(idojaras_lekerdez('Pécs'))
print(idojaras_lekerdez('Miskolc'))

{"hofok": 18, "allapot": "napos", "paratartalom": 45}
{"hiba": "Miskolc nem található"}


### 2. lépés – Az eszköz definiálása az API számára

Az LLM-nek meg kell mondanunk, milyen eszközök állnak rendelkezésre.
Ez egy **JSON séma** – pontosan leírja a függvény nevét, leírását és paramétereit.

Ez a séma **nem** Python kód – ez az LLM számára szóló "használati útmutató".

In [ ]:
eszkozok = [
    {
        'type': 'function',
        'function': {
            'name': 'idojaras_lekerdez',                          # függvény neve
            'description': 'Lekérdezi egy magyar város aktuális időjárását.',  # leírás
            'parameters': {                                       # paraméterek sémája
                'type': 'object',
                'properties': {
                    'varos': {
                        'type': 'string',
                        'description': 'A város neve (pl. Pécs, Budapest)'
                    }
                },
                'required': ['varos']                             # kötelező paraméter
            }
        }
    }
]

print('Eszköz definiálva:', eszkozok[0]['function']['name'])

Eszköz definiálva: idojaras_lekerdez


### 3. lépés – API hívás az eszközzel

Elküldjük a kérdést az LLM-nek, és megadjuk az elérhető eszközöket a `tools` paraméterben.
Ha az LLM úgy dönt, hogy eszközre van szükség, **nem szöveget**, hanem **tool_call-t** ad vissza.

In [ ]:
# A modell meghívása a Chat Completions API segítségével.
# Ez egy kérés az LLM felé, amely eldöntheti:
# 1. közvetlenül válaszol szöveggel
# 2. vagy egy eszközt (tool / function) szeretne meghívni.

valasz = client.chat.completions.create(

    # A használni kívánt nyelvi modell neve.
    # Például: "gpt-4.1", "gpt-4o", stb.
    # A MODELL változó általában a program elején van definiálva.
    model=MODELL,

    # A beszélgetési előzmények listája.
    # A chat API mindig egy üzenetlistát vár.
    messages=[

        # A felhasználó aktuális kérdése.
        # role: jelzi a beszélő szerepét (user / assistant / tool)
        # content: maga az üzenet szövege
        {
            'role': 'user',
            'content': 'Milyen idő van most Pécsett?'
        }

    ],

    # A modell számára elérhető eszközök listája.
    # Ezek olyan függvények, amelyeket a modell kérhet,
    # ha külső adatot kell lekérnie (pl. időjárás API).
    #
    # A modell tehát:
    # - látja a tool definíciókat
    # - eldöntheti, hogy használja-e őket
    tools=eszkozok
)


# A modell válaszának feldolgozása
# ---------------------------------------------------------

# A ChatCompletion válasz több "choice"-ot tartalmazhat,
# de általában az elsőt használjuk.
#
# A struktúra nagyjából így néz ki:
#
# valasz
#   └── choices
#         └── [0]
#              └── message
#
# Innen vesszük ki az assistant üzenetet.
uzenet = valasz.choices[0].message


# Kiírjuk a modell által adott szöveges választ.
#
# FONTOS:
# Ha a modell tool-t akar hívni,
# akkor a content gyakran None lesz.
print('Szöveges válasz:', uzenet.content)


# Kiírjuk, hogy a modell szeretne-e eszközt hívni.
#
# Ha None vagy üres lista → nincs eszközhívás.
#
# Ha van benne elem, akkor a modell egy vagy több
# tool_call objektumot adott vissza, például:
#
# [
#   {
#     id: "call_123",
#     function: {
#         name: "idojaras_lekerdez",
#         arguments: '{"varos":"Pécs"}'
#     }
#   }
# ]
#
# Ezeket a Python-kódnak kell végrehajtania.
print('Eszközhívások:', uzenet.tool_calls)

Szöveges válasz: None
Eszközhívások: [ChatCompletionMessageFunctionToolCall(id='fc_22b0f045-1817-4f01-805a-dae6034903e7', function=Function(arguments='{"varos":"Pécs"}', name='idojaras_lekerdez'), type='function')]


### 4. lépés – A tool_call vizsgálata

Az LLM nem szöveggel válaszolt, hanem egy **eszközhívási kérelemmel**.
Nézzük meg a részleteit: melyik függvényt hívná, milyen paraméterekkel.

In [ ]:
# A modell válaszában található eszközhívások listájából
# kivesszük az első elemet.
#
# A tool_calls egy lista, mert a modell egyszerre több
# eszközt is kérhet.
#
# Például:
# uzenet.tool_calls = [tool_call1, tool_call2, ...]
#
# A [0] index tehát az első eszközhívást jelenti.
tool_call = uzenet.tool_calls[0]   # az első (és most egyetlen) eszközhívás


# Kiírjuk az eszközhívás egyedi azonosítóját.
#
# Ez az ID nagyon fontos, mert amikor visszaküldjük
# a függvény eredményét a modellnek, akkor ezzel
# az azonosítóval kell jelezni, hogy melyik tool_call-hoz
# tartozik a válasz.
#
# Példa:
# call_abc123xyz
print('Eszközhívás ID:', tool_call.id)


# Kiírjuk a modell által meghívni kívánt függvény nevét.
#
# Ez egy szöveg (string), amely megegyezik azzal a névvel,
# amit a tool definícióban megadtunk.
#
# Példa:
# "idojaras_lekerdez"
#
# Ezt a nevet használjuk arra, hogy a Python programban
# kikeressük a megfelelő függvényt egy szótárból.
print('Függvény neve:', tool_call.function.name)


# Kiírjuk a függvény argumentumait.
#
# FONTOS:
# Ez NEM Python szótár!
#
# A modell JSON formátumú SZÖVEGKÉNT adja vissza.
#
# Példa:
# '{"varos":"Pécs"}'
#
# Ezért kell majd JSON-ból Python szótárrá alakítani:
#
# args = json.loads(tool_call.function.arguments)
#
# Ezután már így használható:
#
# idojaras_lekerdez(**args)
#
print('Argumentumok:', tool_call.function.arguments)   # JSON szöveg!

Eszközhívás ID: fc_22b0f045-1817-4f01-805a-dae6034903e7
Függvény neve: idojaras_lekerdez
Argumentumok: {"varos":"Pécs"}


### 5. lépés – A függvény végrehajtása

Az LLM megmondta, mit kell hívni – most **a mi kódunk** hajtja végre.
Az argumentumokat `json.loads()`-zal parse-oljuk, majd meghívjuk a függvényt.

In [ ]:
# Az LLM által kért argumentumok kinyerése
# ------------------------------------------------------------
# A modell a függvény paramétereit JSON formátumú SZÖVEGKÉNT adja vissza.
#
# Példa:
# tool_call.function.arguments = '{"varos":"Pécs"}'
#
# Ez még nem Python szótár, ezért át kell alakítani.

argumentumok = json.loads(tool_call.function.arguments)

# A json.loads() függvény a JSON szöveget Python objektummá alakítja.
#
# Példa átalakulás:
#
# '{"varos":"Pécs"}'
#
# ↓
#
# {'varos': 'Pécs'}
#
# Innentől már normál Python szótárként használható.

print('Kinyert argumentumok:', argumentumok)



# A függvény végrehajtása
# ------------------------------------------------------------
# Most meghívjuk azt a Python függvényt,
# amelyet a modell kért.
#
# A függvény neve ebben az esetben:
# idojaras_lekerdez()

# Az argumentumok szótárból kivesszük a szükséges paramétert.
#
# argumentumok = {'varos': 'Pécs'}
#
# ezért:
# argumentumok['varos']  →  'Pécs'

fuggveny_eredmeny = idojaras_lekerdez(argumentumok['varos'])


# A függvény visszatérési értékét eltároljuk a
# fuggveny_eredmeny változóban.
#
# Például a függvény visszaadhat ilyesmit:
#
# "Pécsett jelenleg 18°C van, enyhén felhős idővel."

print('Függvény eredménye:', fuggveny_eredmeny)

Kinyert argumentumok: {'varos': 'Pécs'}
Függvény eredménye: {"hofok": 18, "allapot": "napos", "paratartalom": 45}


### 6. lépés – Az eredmény visszaküldése az LLM-nek

Ez a legfontosabb lépés! Az eredményt **visszaküldjük** az LLM-nek egy `tool` szerepkörű üzenetben.
Az LLM ezután természetes nyelven válaszol a felhasználónak.

Az üzenettörténet így épül fel:
```
1. user: "Milyen idő van Pécsett?"
2. assistant: tool_call (idojaras_lekerdez, varos=Pécs)
3. tool: {"hofok": 18, "allapot": "napos", ...}
4. assistant: "Pécsett jelenleg 18°C van..."   ← ezt kapjuk most
```

In [ ]:
# Az üzenettörténet összeállítása a teljes ciklussal
uzenetek = [
    {'role': 'user', 'content': 'Milyen idő van most Pécsett?'},
    uzenet,   # az LLM tool_call válasza (assistant szerepkör)
    {
        'role': 'tool',                        # eszköz eredmény szerepkör
        'tool_call_id': tool_call.id,          # melyik hívásra válaszolunk
        'content': fuggveny_eredmeny           # a függvény visszatérési értéke
    }
]

# Második API hívás – most az LLM már ismeri az időjárást
vegso_valasz = client.chat.completions.create(
    model=MODELL,
    messages=uzenetek
)

print(vegso_valasz.choices[0].message.content)

Jelenleg Pécsett 18 °C körüli hőmérséklet, napos az idő és a páratartalom körülbelül 45 %. Enjoy your day!


# Elérhető függvények szótára: név → függvény
elerheto_fuggvenyek = {
    'idojaras_lekerdez': idojaras_lekerdez
}

def kerdez_eszkozzel(user_uzenet, eszkozok, fuggvenyek):
    """Teljes eszközhívási ciklus: kérdés → tool call → végrehajtás → válasz."""
    uzenetek = [{'role': 'user', 'content': user_uzenet}]

    # 1. lépés: kérdés küldése
    valasz = client.chat.completions.create(
        model=MODELL,
        messages=uzenetek,
        tools=eszkozok
    )
    uzenet = valasz.choices[0].message

    # Ha nincs tool_call, sima szöveges válasz
    if not uzenet.tool_calls:
        return uzenet.content

    # 2. lépés: tool_call(ok) végrehajtása
    uzenetek.append(uzenet)   # az assistant tool_call üzenete

    for tc in uzenet.tool_calls:
        fv_nev = tc.function.name
        fv_args = json.loads(tc.function.arguments)

        # Üres vagy hibás argumentumok tisztítása
        # Egyes modellek paraméter nélküli függvényekhez is küldenek szemetet (pl. {'': {}})
        fv_args = {k: v for k, v in fv_args.items() if k}  # üres kulcsok eltávolítása

        print(f'  Eszközhívás: {fv_nev}({fv_args})')

        # Végrehajtás
        eredmeny = fuggvenyek[fv_nev](**fv_args)

        # Eredmény visszaküldése
        uzenetek.append({
            'role': 'tool',
            'tool_call_id': tc.id,
            'content': eredmeny
        })

    # 3. lépés: végső válasz
    vegso = client.chat.completions.create(
        model=MODELL,
        messages=uzenetek
    )
    return vegso.choices[0].message.content

print('Függvény kész')

In [ ]:
# Elérhető függvények szótára: név → valódi Python függvény
# Ez azért kell, mert a modell a függvény nevét szövegként adja vissza,
# nekünk pedig ebből ki kell keresnünk a ténylegesen meghívható függvényt.
elerheto_fuggvenyek = {
    'idojaras_lekerdez': idojaras_lekerdez
}

def kerdez_eszkozzel(user_uzenet, eszkozok, fuggvenyek):
    """
    Teljes eszközhívási ciklus kezelése.

    A függvény működése röviden:
    1. elküldi a felhasználó kérdését a modellnek,
    2. a modell eldöntheti, hogy akar-e eszközt/függvényt hívni,
    3. ha igen, a Python-kód végrehajtja a megfelelő függvényt,
    4. a függvény eredményét visszaküldi a modellnek,
    5. végül a modell elkészíti a végleges természetes nyelvű választ.

    Paraméterek:
    - user_uzenet: a felhasználó kérdése szövegként
    - eszkozok: a modell számára átadott tool definíciók listája
    - fuggvenyek: a meghívható Python függvények szótára
                  pl. {'idojaras_lekerdez': idojaras_lekerdez}

    Visszatérési érték:
    - a modell végső szöveges válasza
    """

    # A beszélgetés üzenetlistáját hozzuk létre.
    # Az OpenAI chat API mindig üzenetek listájával dolgozik.
    # Kezdetben csak a felhasználó aktuális kérdése szerepel benne.
    uzenetek = [
        {
            'role': 'user',
            'content': user_uzenet
        }
    ]

    # ------------------------------------------------------------------
    # 1. LÉPÉS: Az első kérés elküldése a modellnek
    # ------------------------------------------------------------------
    # A modell itt még nem ad végleges választ feltétlenül.
    # Lehet, hogy úgy dönt, hogy előbb egy eszközt (tool) kell meghívni.
    valasz = client.chat.completions.create(
        model=MODELL,        # Melyik modellt használjuk
        messages=uzenetek,   # Az eddigi beszélgetés
        tools=eszkozok       # Az elérhető eszközök definíciói
    )

    # Az API válaszából kivesszük az első choice első üzenetét.
    # Ez lesz az assistant válasza, ami lehet:
    # - sima szöveges válasz
    # - vagy tool_call(oka)t tartalmazó válasz
    uzenet = valasz.choices[0].message

    # ------------------------------------------------------------------
    # Ha a modell NEM akar eszközt hívni
    # ------------------------------------------------------------------
    # Ebben az esetben a válasz közvetlenül szöveges tartalomként érkezik.
    # Nincs további teendő, egyszerűen visszaadjuk.
    if not uzenet.tool_calls:
        return uzenet.content

    # ------------------------------------------------------------------
    # 2. LÉPÉS: A modell által kért tool_call(ok) végrehajtása
    # ------------------------------------------------------------------

    # Nagyon fontos: hozzáadjuk az assistant tool_call üzenetét
    # a beszélgetési előzményekhez.
    # Erre azért van szükség, mert a második API-hívásnál a modellnek
    # látnia kell, hogy korábban milyen eszközhívást kezdeményezett.
    uzenetek.append(uzenet)

    # A modell akár több tool_call-t is visszaadhat egyszerre,
    # ezért végigmegyünk mindegyiken.
    for tc in uzenet.tool_calls:

        # A meghívandó függvény neve, például: "idojaras_lekerdez"
        fv_nev = tc.function.name

        # A modell az argumentumokat JSON formátumú szövegként küldi vissza.
        # Ezt Python szótárrá alakítjuk.
        fv_args = json.loads(tc.function.arguments)

        # --------------------------------------------------------------
        # Hibás / felesleges argumentumok tisztítása
        # --------------------------------------------------------------
        # Egyes modellek paraméter nélküli függvény esetén is küldhetnek
        # értelmetlen struktúrát, például {"": {}} vagy hasonlót.
        #
        # Ez a sor eltávolít minden olyan kulcsot, amely üres string.
        # Így elkerülhető, hogy a függvényhívás hibára fusson.
        fv_args = {k: v for k, v in fv_args.items() if k}

        # Konzolos naplózás:
        # kiírjuk, hogy a modell pontosan milyen függvényt akar meghívni,
        # és milyen argumentumokkal.
        # Ez hibakereséshez nagyon hasznos.
        print(f'  Eszközhívás: {fv_nev}({fv_args})')

        # --------------------------------------------------------------
        # A függvény tényleges végrehajtása
        # --------------------------------------------------------------
        # A fuggvenyek szótárból kikeressük a megfelelő Python függvényt
        # a neve alapján, majd meghívjuk azt a kapott argumentumokkal.
        #
        # A **fv_args szótárkicsomagolás:
        # pl. {"varos": "Budapest"} -> fuggveny(varos="Budapest")
        eredmeny = fuggvenyek[fv_nev](**fv_args)

        # --------------------------------------------------------------
        # A függvény eredményének visszaadása a modellnek
        # --------------------------------------------------------------
        # A tool szerepű üzenetben visszaküldjük:
        # - melyik tool_call-hoz tartozik az eredmény (tool_call_id)
        # - mi lett a függvény válasza (content)
        #
        # Ezt a modell a következő lépésben felhasználja arra,
        # hogy emberi nyelvű végső választ állítson össze.
        uzenetek.append({
            'role': 'tool',
            'tool_call_id': tc.id,
            'content': eredmeny
        })

    # ------------------------------------------------------------------
    # 3. LÉPÉS: A végső válasz elkészíttetése a modellel
    # ------------------------------------------------------------------
    # Most már az üzenetlistában benne van:
    # - a felhasználó kérdése
    # - az assistant tool_call üzenete
    # - a tool válasz(ok)
    #
    # Ezek alapján a modell elkészíti a végleges természetes nyelvű választ.
    vegso = client.chat.completions.create(
        model=MODELL,
        messages=uzenetek
    )

    # Visszaadjuk a végső szöveges választ
    return vegso.choices[0].message.content

# Egyszerű visszajelzés a konzolra, hogy a függvény definiálása megtörtént
print('Függvény kész')

Függvény kész


### 8. lépés – Tesztelés különböző kérdésekkel

Próbáljuk ki a függvényt! Az LLM automatikusan eldönti, mikor kell eszközt hívni.

In [ ]:
# Eszközt igénylő kérdés
print('--- Időjárás kérdés ---')
print(kerdez_eszkozzel('Milyen idő van Szegeden?', eszkozok, elerheto_fuggvenyek))

--- Időjárás kérdés ---
  Eszközhívás: idojaras_lekerdez({'varos': 'Szeged'})
Szegeden jelenleg 20 °C körüli hőmérséklet van, napos az idő, a páratartalom pedig körülbelül 40 %. 🌞


In [ ]:
# Eszközt NEM igénylő kérdés – az LLM nem hívja az eszközt
print('--- Általános kérdés ---')
print(kerdez_eszkozzel('Mi a Python programozási nyelv?', eszkozok, elerheto_fuggvenyek))

--- Általános kérdés ---
A Python egy magas szintű, általános célú programozási nyelv, amelyet Guido van Rossum fejlesztett ki 1991-ben. Főbb jellemzői:

* **Olvashatóság** – a szintaxisa egyszerű és tiszta, a kódblokkokat behúzással (indentálással) jelöli, így a programok könnyen érthetőek.
* **Interaktív** – a Python interpreterrel közvetlenül kipróbálhatod a kódot, ami gyors prototípus‑készítést tesz lehetővé.
* **Dinamikus típuskezelés** – a változók típusa futásidőben kerül meghatározásra, így nem kell előre deklarálni.
* **Gazdag standard könyvtár** – beépített modulok (pl. `os`, `datetime`, `json`, `re`) széles körű feladatok megoldására.
* **Kiterjeszthetőség** – C/C++‑ban írt modulokkal vagy külső csomagokkal (pip‑en keresztül) könnyen bővíthető.
* **Platformfüggetlen** – ugyanaz a kód futtatható Windows, macOS és Linux rendszereken is.

**Alkalmazási területek**

* Webfejlesztés (Django, Flask)
* Adattudomány és gépi tanulás (NumPy, pandas, scikit‑learn, TensorFlow, PyTorch)


In [ ]:
# Ismeretlen város – az eszköz hibaüzenetet ad
print('--- Ismeretlen város ---')
print(kerdez_eszkozzel('Milyen idő van Miskolcon?', eszkozok, elerheto_fuggvenyek))

--- Ismeretlen város ---
  Eszközhívás: idojaras_lekerdez({'varos': 'Miskolc'})
Sajnos a rendelkezésemre álló adatokban Miskolc időjárási információja nem szerepel. Ha más városra vagy általános időjárási információra van szükséged, kérlek jelezd!


---
## 8. Több eszköz együtt

Az LLM egyszerre **több eszközt** is ismerhet – és a kérdés alapján kiválasztja a megfelelőt.
Készítünk két további szimulált eszközt:

- `valuta_atvaltas()` – valuták közötti átváltás
- `naptar_lekerdez()` – mai nap és névnap lekérdezése

### 1. lépés – Új szimulált függvények

Mindkét függvény egyszerű szimulált adatokat ad vissza JSON szövegként.

In [ ]:
def valuta_atvaltas(osszeg, forras, cel):
    """Szimulált valutaátváltó."""
    arfolyamok = {
        ('HUF', 'EUR'): 0.0025,
        ('EUR', 'HUF'): 395.0,
        ('HUF', 'USD'): 0.0027,
        ('USD', 'HUF'): 370.0,
        ('EUR', 'USD'): 1.08,
        ('USD', 'EUR'): 0.93,
    }
    kulcs = (forras.upper(), cel.upper())
    if kulcs in arfolyamok:
        eredmeny = osszeg * arfolyamok[kulcs]
        return json.dumps({'osszeg': osszeg, 'forras': forras, 'cel': cel,
                           'eredmeny': round(eredmeny, 2), 'arfolyam': arfolyamok[kulcs]},
                          ensure_ascii=False)
    return json.dumps({'hiba': f'{forras}→{cel} átváltás nem elérhető'}, ensure_ascii=False)


def naptar_lekerdez():
    """Szimulált naptár – mai nap adatai."""
    return json.dumps({
        'datum': '2026-03-16',
        'nap': 'hétfő',
        'nevnap': ['Henrietta'],
        'unnep': None
    }, ensure_ascii=False)


# Teszt
print(valuta_atvaltas(1000, 'EUR', 'HUF'))
print(naptar_lekerdez())

{"osszeg": 1000, "forras": "EUR", "cel": "HUF", "eredmeny": 395000.0, "arfolyam": 395.0}
{"datum": "2026-03-16", "nap": "hétfő", "nevnap": ["Henrietta"], "unnep": null}


### 2. lépés – A három eszköz definiálása együtt

Most mindhárom eszközt egyetlen listában adjuk meg az API-nak.
Az LLM a kérdés alapján **automatikusan kiválasztja** a megfelelőt.

In [ ]:
minden_eszkoz = [
    {
        'type': 'function',
        'function': {
            'name': 'idojaras_lekerdez',
            'description': 'Lekérdezi egy magyar város aktuális időjárását.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'varos': {'type': 'string', 'description': 'A város neve'}
                },
                'required': ['varos']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'valuta_atvaltas',
            'description': 'Átváltja az összeget egyik valutából a másikba (HUF, EUR, USD).',
            'parameters': {
                'type': 'object',
                'properties': {
                    'osszeg': {'type': 'number', 'description': 'Az átváltandó összeg'},
                    'forras': {'type': 'string', 'description': 'Forrás valuta (pl. HUF)'},
                    'cel':    {'type': 'string', 'description': 'Cél valuta (pl. EUR)'}
                },
                'required': ['osszeg', 'forras', 'cel']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'naptar_lekerdez',
            'description': 'Visszaadja a mai nap adatait: dátum, nap neve, névnap.',
            'parameters': {
                'type': 'object',
                'properties': {},
                'required': []
            }
        }
    }
]

# Az elérhető függvények szótárát is bővítjük
minden_fuggveny = {
    'idojaras_lekerdez': idojaras_lekerdez,
    'valuta_atvaltas':   valuta_atvaltas,
    'naptar_lekerdez':   naptar_lekerdez
}

print(f'{len(minden_eszkoz)} eszköz definiálva:', [e["function"]["name"] for e in minden_eszkoz])

3 eszköz definiálva: ['idojaras_lekerdez', 'valuta_atvaltas', 'naptar_lekerdez']


### 3. lépés – Tesztelés: az LLM választja ki az eszközt

Három különböző kérdés – mindegyik más eszközt igényel.
A `kerdez_eszkozzel()` függvény automatikusan kezeli a teljes ciklust.

In [ ]:
print('=== Időjárás kérdés ===')
print(kerdez_eszkozzel('Milyen idő van Debrecenben?', minden_eszkoz, minden_fuggveny))
print()

=== Időjárás kérdés ===
  Eszközhívás: idojaras_lekerdez({'varos': 'Debrecen'})
Debrecenben jelenleg 12 °C körüli hőmérséklet van, esős az idő, a páratartalom pedig 80 %. 🌧️



In [ ]:
print('=== Valuta kérdés ===')
print(kerdez_eszkozzel('Mennyit ér 50000 forint euróban?', minden_eszkoz, minden_fuggveny))
print()

=== Valuta kérdés ===
  Eszközhívás: valuta_atvaltas({'cel': 'EUR', 'forras': 'HUF', 'osszeg': 50000})
50000 forint jelenleg körülbelül **125,00 euró**-nak felel meg (az átváltási árfolyam 1 HUF ≈ 0,0025 EUR).



In [ ]:
print('=== Naptár kérdés ===')
print(kerdez_eszkozzel('Milyen nap van ma? Kinek van névnapja?', minden_eszkoz, minden_fuggveny))
print()

=== Naptár kérdés ===
  Eszközhívás: naptar_lekerdez({})
Ma 2026. március 16. van, hétfő. A mai névnapok: **Henrietta**. (A naptár szerint ez az egyetlen névnap, de egyes források szerint több név is előfordulhat.)



In [ ]:
print('=== Általános kérdés (nincs eszközhívás) ===')
print(kerdez_eszkozzel('Ki írta az Egri csillagokat?', minden_eszkoz, minden_fuggveny))

=== Általános kérdés (nincs eszközhívás) ===
Az **Egri csillagok** című regényt **Gárdonyi Géza** írta. 1899‑ben jelent meg először, és azóta a magyar irodalom egyik klasszikusának számít.


---
## Összefoglalás

| Fejezet | Téma | Kulcstechnika |
|---|---|---|
| 1. | OpenAI-kompatibilis API | `base_url` váltás → bármely szolgáltató |
| 2. | OpenRouter beállítás | `OpenAI(base_url=..., api_key=...)` |
| 3. | Chat completion | `client.chat.completions.create()` |
| 4. | Szerepek (system/user/assistant) | System prompt → viselkedés szabályozás |
| 5. | JSON mód | `response_format={"type": "json_object"}` |
| 6–7. | Eszközhívás (Tool Calling) | `tools=[...]`, `tool_calls`, `role: tool` |
| 8. | Több eszköz együtt | Az LLM automatikusan választ |

### Gemini API vs. OpenAI-kompatibilis API összehasonlítás

| | Gemini natív | OpenAI-kompatibilis |
|---|---|---|
| Csomag | `google-generativeai` | `openai` |
| Hívás | `model.generate_content(prompt)` | `client.chat.completions.create(messages=...)` |
| JSON mód | `response_mime_type` + `response_schema` | `response_format={"type": "json_object"}` |
| Eszközhívás | `function_declarations` | `tools=[{"type": "function", ...}]` |
| Hordozhatóság | csak Google | OpenAI, OpenRouter, Groq, Ollama, stb. |

**Következő óra:** Többkörös beszélgetések, haladó eszközhívások, komplex ügynök (agent) építés.